# Getting started with MemsArrayH5 objects

The `MemsArrayH5` class allows getting signals from MemsArray saved in a H5 file with data formated as MuH5 format

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from megamicros.log import log
from megamicros.core.h5 import MemsArrayH5

log.setLevel( "INFO" )

# Choose a file where some H5 files are stored
DIRECTORY = '/Users/brunogas/Data/muh5/'
#DIRECTORY = '/Users/brunogas/Documents/Dev/Bimea/megamicros-cpp/build/'

# Choose a H5 file
#FILENAME = DIRECTORY + 'mu5h-20220630-064820.h5'
FILENAME = DIRECTORY + 'mu5h-20231107-082957.h5'


#FILENAME = DIRECTORY + 'mu5h-20220124-002506.h5'
#FILENAME = DIRECTORY + 'mu5h-20220720-160341.h5'
#FILENAME = DIRECTORY + 'muh5-20231021-162010.h5'

## Starting with simple signals

One has only to specify the file or directory name where to find H5 files.

The ``MemsArrayH5`` open the input H5 file or the first H5 file of the directory list and populates the antenna parameters with metadata read from the file. An exception is raised if file or directory are not found.

In [ ]:
# Define the antenna
antenna = MemsArrayH5( 
    filename=FILENAME
)

You can get now some Meta informations regarding the file

In [ ]:
print( f"Sampling frequency: {antenna.sampling_frequency}Hz" )
print( f"Available MEMs number: {antenna.available_mems_number}" )
print( f"Whether counter is available or not: {antenna.counter}" )

## Getting and plotting some MEMs signals

You can select some MEMs you would like to plot and get signals on a given range time (10 seconds in this example)

In [ ]:
# Run antenna
antenna.run(
    mems = [3, 4],
    duration=4,
    counter_skip = True,
    datatype='int32'
)

# Init a np.ndarray
signals = np.ndarray( (0, antenna.channels_number ) )

# Get signals
for data in antenna:
    signals = np.concatenate( ( signals, data ), axis=0 )

# waiting for the end of the running thread is mandatory
antenna.wait()
print( f"exit from loop. Signal shape is: {np.shape( signals )}" )

Here is the plotting program

In [ ]:
# plot signals
time = np.array( range( np.size(signals,0) ) )/antenna.sampling_frequency
fig, axs = plt.subplots( antenna.channels_number )
fig.suptitle('Channels activity')	
for s in range( antenna.channels_number ):
    axs[s].plot( time, signals[:,s] )
    axs[s].set( xlabel='time in seconds', ylabel='channel %d' % s )

plt.show()

## Saving signals as wave file

In [ ]:
import wave

WAV_FILENAME = 'titi.wav'

# 2 seconds run, getting signals from MEMs 1 and 2
antenna.run(
    mems = [5, 6],
    duration=5,
    counter_skip = True,
    datatype='int32'
)

with  wave.open( WAV_FILENAME, mode='wb' ) as wavfile:
    wavfile.setnchannels(2)
    wavfile.setsampwidth(2)
    wavfile.setframerate( antenna.sampling_frequency )

    # Get signals
    for data in antenna:
        signal = data >> 4
        wavfile.writeframesraw( np.int16( np.reshape( signal, np.size( signal ), order='C' ) ) )

# waiting for the end of the running thread is mandatory
antenna.wait()

## Hearing signal with *pyaudio* library

In [ ]:
import pyaudio

FRAME_LENGTH = 512

# Instantiate PyAudio and initialize PortAudio system resources (1)
p = pyaudio.PyAudio()

# Open stream
stream = p.open(
    format = pyaudio.paFloat32,
    channels = 2,
    rate = int( antenna.sampling_frequency ),
    output=True,
    frames_per_buffer=FRAME_LENGTH,
)

# Start running the remote Megamicros system
antenna.run( 
    mems=[1, 2],
    duration=10,
    frame_length=FRAME_LENGTH,
    counter_skip = True,
    datatype='int32'
)

# Get signals
transfers_counter = 0
for data in antenna:
    signal = data >> 4

    # convert into float and normalize with MEMs sensibility
    data = ( data.astype( np.float32 ) * antenna.sensibility )

    # write into audio stream
    stream.write( data.tobytes( order='C'), num_frames=FRAME_LENGTH )
    transfers_counter += 1

# Close stream and release PortAudio system resources (5)
stream.close()            
p.terminate()

antenna.wait()